In [ ]:
from pathlib import Path
import pandas as pd

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

INPUT_ROOT = Path("/kaggle/input")

search_parents = [
    INPUT_ROOT,
    INPUT_ROOT / "competitions",
]

dataset_roots = [
    child
    for parent in search_parents
    if parent.exists()
    for child in parent.iterdir()
    if child.is_dir()
    and (child / "train.csv").exists()
    and (child / "train_series.csv").exists()
]

if len(dataset_roots) != 1:
    available_inputs = {
        str(parent): sorted(
            child.name
            for child in parent.iterdir()
            if child.is_dir()
        )
        for parent in search_parents
        if parent.exists()
    }

    raise RuntimeError(
        "Expected exactly one competition dataset root. "
        f"Candidates: {dataset_roots}. "
        f"Available inputs: {available_inputs}"
    )

DATA_ROOT = dataset_roots[0]

train = pd.read_csv(
    DATA_ROOT / "train.csv",
    dtype={"StudyInstanceUID": str},
)

required_columns = {
    "StudyInstanceUID",
    "Report",
    *TARGETS,
}

missing_columns = required_columns - set(train.columns)

if missing_columns:
    raise RuntimeError(
        f"train.csv is missing columns: {sorted(missing_columns)}"
    )

gold_mask = train[TARGETS].notna().all(axis=1)

partial_label_mask = (
    train[TARGETS].notna().any(axis=1)
    & ~gold_mask
)

unlabeled_mask = ~train[TARGETS].notna().any(axis=1)

reports = train["Report"].fillna("").astype(str)

report_audit = pd.DataFrame(
    {
        "characters": reports.str.len(),
        "words": reports.str.split().str.len(),
        "lines": reports.str.count(r"\n") + 1,
    }
)

print("Dataset root:", DATA_ROOT)
print("Studies:", len(train))
print("Gold-labeled studies:", int(gold_mask.sum()))
print("Partially labeled studies:", int(partial_label_mask.sum()))
print("Unlabeled studies:", int(unlabeled_mask.sum()))
print(
    "Missing or empty reports:",
    int(reports.str.strip().eq("").sum()),
)
print(
    "Duplicate study IDs:",
    int(train["StudyInstanceUID"].duplicated().sum()),
)
print(
    "Exact duplicate reports:",
    int(reports.duplicated().sum()),
)

print("\nReport length distribution:")

display(
    report_audit.describe(
        percentiles=[
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
        ]
    ).T
)

print("\nGold versus unlabeled report lengths:")

display(
    report_audit.assign(
        supervision=gold_mask.map(
            {
                True: "gold",
                False: "report_only",
            }
        )
    )
    .groupby("supervision")
    .agg(
        studies=("characters", "size"),
        median_characters=("characters", "median"),
        median_words=("words", "median"),
        median_lines=("lines", "median"),
    )
)

In [ ]:
import re

normalized_reports = (
    reports
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

def duplicate_summary(values):
    counts = values.value_counts(dropna=False)
    duplicate_groups = counts[counts > 1]

    return {
        "unique_reports": int(counts.size),
        "duplicate_groups": int(duplicate_groups.size),
        "studies_in_duplicate_groups": int(duplicate_groups.sum()),
        "duplicate_rows_after_first": int(
            (duplicate_groups - 1).sum()
        ),
        "largest_group": int(
            duplicate_groups.max()
            if len(duplicate_groups)
            else 1
        ),
    }

duplicate_table = pd.DataFrame(
    {
        "exact": duplicate_summary(reports),
        "whitespace_case_normalized": duplicate_summary(
            normalized_reports
        ),
    }
).T

print("Duplicate-report structure:")
display(duplicate_table)

duplicate_frame = pd.DataFrame(
    {
        "normalized_report": normalized_reports,
        "is_gold": gold_mask,
    }
)

group_supervision = (
    duplicate_frame
    .groupby("normalized_report")
    .agg(
        studies=("is_gold", "size"),
        gold_studies=("is_gold", "sum"),
    )
)

duplicate_group_supervision = group_supervision[
    group_supervision["studies"] > 1
]

cross_supervision_groups = duplicate_group_supervision[
    (duplicate_group_supervision["gold_studies"] > 0)
    & (
        duplicate_group_supervision["gold_studies"]
        < duplicate_group_supervision["studies"]
    )
]

print(
    "Duplicate groups containing both gold and report-only studies:",
    len(cross_supervision_groups),
)
print(
    "Report-only studies sharing a normalized report with a gold study:",
    int(
        (
            cross_supervision_groups["studies"]
            - cross_supervision_groups["gold_studies"]
        ).sum()
    ),
)

gold_with_report = (
    train.loc[
        gold_mask,
        [*TARGETS],
    ]
    .assign(
        normalized_report=normalized_reports[gold_mask].values
    )
)

gold_group_sizes = (
    gold_with_report
    .groupby("normalized_report")
    .size()
)

gold_duplicate_reports = set(
    gold_group_sizes[
        gold_group_sizes > 1
    ].index
)

gold_conflict_groups = 0

for report_key in gold_duplicate_reports:
    group = gold_with_report[
        gold_with_report["normalized_report"] == report_key
    ]

    if group[TARGETS].nunique().max() > 1:
        gold_conflict_groups += 1

print(
    "Gold duplicate-report groups:",
    len(gold_duplicate_reports),
)
print(
    "Gold duplicate groups with conflicting labels:",
    gold_conflict_groups,
)

section_patterns = {
    "history": r"(?im)^\s*(clinical\s+history|history|indication)\s*:",
    "comparison": r"(?im)^\s*(comparison|comparisons)\s*:",
    "technique": r"(?im)^\s*(technique|exam|procedure)\s*:",
    "findings": r"(?im)^\s*(findings|finding)\s*:",
    "impression": r"(?im)^\s*(impression|conclusion)\s*:",
}

section_rows = []

for section, pattern in section_patterns.items():
    present = reports.str.contains(
        pattern,
        regex=True,
        na=False,
    )

    section_rows.append(
        {
            "section": section,
            "studies": int(present.sum()),
            "percentage": round(
                100 * present.mean(),
                2,
            ),
            "gold_studies": int(
                (present & gold_mask).sum()
            ),
        }
    )

print("\nRecognized report sections:")
display(
    pd.DataFrame(section_rows)
    .set_index("section")
)

non_ascii_fraction = reports.map(
    lambda text: (
        sum(ord(character) > 127 for character in text)
        / max(len(text), 1)
    )
)

print(
    "Reports containing any non-ASCII characters:",
    int((non_ascii_fraction > 0).sum()),
)
print(
    "Reports with more than 5% non-ASCII characters:",
    int((non_ascii_fraction > 0.05).sum()),
)

In [ ]:
import unicodedata

SCRIPT_NAMES = [
    "LATIN",
    "CYRILLIC",
    "ARABIC",
    "GREEK",
    "HEBREW",
    "HANGUL",
    "HIRAGANA",
    "KATAKANA",
    "CJK",
    "THAI",
    "DEVANAGARI",
]

def report_script_counts(text):
    counts = {
        script: 0
        for script in SCRIPT_NAMES
    }
    counts["OTHER_LETTERS"] = 0

    for character in text:
        if not character.isalpha():
            continue

        unicode_name = unicodedata.name(
            character,
            "",
        )

        matched_script = None

        for script in SCRIPT_NAMES:
            if script in unicode_name:
                matched_script = script
                break

        if matched_script is None:
            counts["OTHER_LETTERS"] += 1
        else:
            counts[matched_script] += 1

    return counts

script_counts = pd.DataFrame(
    reports.map(report_script_counts).tolist()
)

letter_totals = script_counts.sum(axis=1)

dominant_script = script_counts.idxmax(axis=1)

dominant_share = (
    script_counts.max(axis=1)
    / letter_totals.clip(lower=1)
)

script_summary = pd.DataFrame(
    {
        "dominant_script": dominant_script,
        "dominant_share": dominant_share,
        "is_gold": gold_mask,
    }
)

dominant_table = (
    script_summary
    .groupby("dominant_script")
    .agg(
        studies=("dominant_script", "size"),
        gold_studies=("is_gold", "sum"),
        median_dominant_share=("dominant_share", "median"),
    )
    .sort_values(
        "studies",
        ascending=False,
    )
)

print("Dominant writing systems:")
display(dominant_table)

print("Total alphabetic characters by writing system:")
display(
    script_counts
    .sum()
    .sort_values(ascending=False)
    .rename("characters")
    .to_frame()
)

mixed_script_mask = dominant_share < 0.90

print(
    "Reports with dominant-script share below 90%:",
    int(mixed_script_mask.sum()),
)
print(
    "Gold reports with dominant-script share below 90%:",
    int((mixed_script_mask & gold_mask).sum()),
)

latin_reports = dominant_script.eq("LATIN")

accented_latin_fraction = reports.map(
    lambda text: (
        sum(
            character.isalpha()
            and ord(character) > 127
            and "LATIN" in unicodedata.name(character, "")
            for character in text
        )
        / max(
            sum(character.isalpha() for character in text),
            1,
        )
    )
)

print(
    "Latin-dominant reports with accented-letter fraction above 1%:",
    int(
        (
            latin_reports
            & (accented_latin_fraction > 0.01)
        ).sum()
    ),
)
print(
    "Latin-dominant reports with accented-letter fraction above 5%:",
    int(
        (
            latin_reports
            & (accented_latin_fraction > 0.05)
        ).sum()
    ),
)

In [ ]:
import numpy as np

SPLIT_SEED = 42
EVALUATION_SIZE = 29
SEARCH_ITERATIONS = 20_000

gold_indices = train.index[gold_mask].to_numpy()

gold_labels = (
    train.loc[
        gold_indices,
        TARGETS,
    ]
    .astype(int)
    .to_numpy()
)

gold_scripts = (
    dominant_script.loc[
        gold_indices
    ]
    .to_numpy()
)

rng = np.random.default_rng(SPLIT_SEED)

best_evaluation_positions = None
best_score = float("inf")

all_positions = np.arange(len(gold_indices))

for _ in range(SEARCH_ITERATIONS):
    evaluation_positions = rng.choice(
        all_positions,
        size=EVALUATION_SIZE,
        replace=False,
    )

    evaluation_position_set = set(
        evaluation_positions.tolist()
    )

    development_positions = np.array(
        [
            position
            for position in all_positions
            if position not in evaluation_position_set
        ]
    )

    development_labels = gold_labels[
        development_positions
    ]

    evaluation_labels = gold_labels[
        evaluation_positions
    ]

    # Every target must contain both classes in both partitions.
    if (
        (development_labels.sum(axis=0) == 0).any()
        or (
            development_labels.sum(axis=0)
            == len(development_labels)
        ).any()
        or (evaluation_labels.sum(axis=0) == 0).any()
        or (
            evaluation_labels.sum(axis=0)
            == len(evaluation_labels)
        ).any()
    ):
        continue

    development_scripts = gold_scripts[
        development_positions
    ]

    evaluation_scripts = gold_scripts[
        evaluation_positions
    ]

    # Preserve at least one Greek and one Cyrillic report in each split.
    required_scripts = [
        "GREEK",
        "CYRILLIC",
    ]

    if any(
        (development_scripts == script).sum() == 0
        or (evaluation_scripts == script).sum() == 0
        for script in required_scripts
    ):
        continue

    development_rates = development_labels.mean(axis=0)
    evaluation_rates = evaluation_labels.mean(axis=0)
    overall_rates = gold_labels.mean(axis=0)

    label_score = (
        np.abs(development_rates - overall_rates).sum()
        + np.abs(evaluation_rates - overall_rates).sum()
    )

    script_score = 0.0

    for script in required_scripts:
        overall_script_rate = (
            gold_scripts == script
        ).mean()

        script_score += abs(
            (development_scripts == script).mean()
            - overall_script_rate
        )

        script_score += abs(
            (evaluation_scripts == script).mean()
            - overall_script_rate
        )

    score = label_score + script_score

    if score < best_score:
        best_score = score
        best_evaluation_positions = (
            evaluation_positions.copy()
        )

if best_evaluation_positions is None:
    raise RuntimeError(
        "Unable to construct a valid gold-label split"
    )

evaluation_position_set = set(
    best_evaluation_positions.tolist()
)

development_positions = np.array(
    [
        position
        for position in all_positions
        if position not in evaluation_position_set
    ]
)

split_assignments = pd.DataFrame(
    {
        "StudyInstanceUID": train.loc[
            gold_indices,
            "StudyInstanceUID",
        ].values,
        "split": [
            (
                "evaluation"
                if position in evaluation_position_set
                else "development"
            )
            for position in all_positions
        ],
    }
)

split_path = Path(
    "/kaggle/working/report_gold_split.csv"
)

split_assignments.to_csv(
    split_path,
    index=False,
)

development_labels = gold_labels[
    development_positions
]

evaluation_labels = gold_labels[
    best_evaluation_positions
]

balance_table = pd.DataFrame(
    {
        "all_positive": gold_labels.sum(axis=0),
        "development_positive": development_labels.sum(axis=0),
        "evaluation_positive": evaluation_labels.sum(axis=0),
    },
    index=TARGETS,
)

script_balance = pd.crosstab(
    pd.Series(
        gold_scripts,
        name="script",
    ),
    pd.Series(
        split_assignments["split"].values,
        name="split",
    ),
)

print("Split seed:", SPLIT_SEED)
print("Development studies:", len(development_positions))
print("Evaluation studies:", len(best_evaluation_positions))
print("Balance objective:", round(best_score, 6))
print("Split saved to:", split_path)

print("\nLabel balance:")
display(balance_table)

print("\nWriting-system balance:")
display(script_balance)

In [ ]:
from pathlib import Path
import importlib.util
import json

import torch
import transformers

EXPECTED_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
EXPECTED_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"

provenance_paths = list(
    Path("/kaggle/input").glob(
        "notebooks/*/*/*/MODEL_PROVENANCE.json"
    )
)

matching_model_dirs = []

for provenance_path in provenance_paths:
    provenance = json.loads(
        provenance_path.read_text()
    )

    if (
        provenance.get("source_repository")
        == EXPECTED_MODEL_ID
        and provenance.get("source_revision")
        == EXPECTED_REVISION
    ):
        matching_model_dirs.append(
            provenance_path.parent
        )

if len(matching_model_dirs) != 1:
    raise RuntimeError(
        "Expected exactly one pinned Qwen model bundle. "
        f"Found: {matching_model_dirs}"
    )

QWEN_MODEL_DIR = matching_model_dirs[0]

required_files = {
    "LICENSE",
    "MODEL_PROVENANCE.json",
    "inventory.csv",
    "config.json",
    "generation_config.json",
    "model.safetensors.index.json",
    "model-00001-of-00003.safetensors",
    "model-00002-of-00003.safetensors",
    "model-00003-of-00003.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
}

missing_files = sorted(
    filename
    for filename in required_files
    if not (QWEN_MODEL_DIR / filename).exists()
)

if missing_files:
    raise RuntimeError(
        f"Qwen bundle is incomplete: {missing_files}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a Kaggle GPU and restart the session."
    )

print("Model directory:", QWEN_MODEL_DIR)
print("Model:", EXPECTED_MODEL_ID)
print("Revision:", EXPECTED_REVISION)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory (GB):",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2,
    ),
)
print(
    "Accelerate installed:",
    importlib.util.find_spec("accelerate") is not None,
)
print("Offline Qwen bundle verification passed.")

In [ ]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_DIR,
    local_files_only=True,
)

report_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_DIR,
    local_files_only=True,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

report_model.eval()
report_model.requires_grad_(False)

torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_start

parameter_count = sum(
    parameter.numel()
    for parameter in report_model.parameters()
)

trainable_count = sum(
    parameter.numel()
    for parameter in report_model.parameters()
    if parameter.requires_grad
)

print("Model class:", type(report_model))
print("Parameters:", f"{parameter_count:,}")
print("Trainable parameters:", trainable_count)
print("Training mode:", report_model.training)
print("Parameter dtype:", next(report_model.parameters()).dtype)
print("Parameter device:", next(report_model.parameters()).device)
print("Tokenizer size:", len(tokenizer))
print("Load time (seconds):", round(load_seconds, 2))
print(
    "GPU memory allocated (GB):",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
)
print(
    "Peak GPU memory allocated (GB):",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
)
print("Offline Qwen load passed.")

In [ ]:
import json
import time

STATUS_VALUES = {
    "present",
    "absent",
    "uncertain",
    "not_mentioned",
}

SEVERITY_VALUES = {
    "none",
    "mild",
    "moderate",
    "severe",
    "unspecified",
}

TARGET_DESCRIPTIONS = {
    "ACL": (
        "Pathology of the anterior cruciate ligament, including "
        "tear, rupture, sprain, degeneration, or abnormal signal."
    ),
    "MCL": (
        "Pathology of the medial collateral ligament, including "
        "tear, rupture, sprain, thickening, or abnormal signal."
    ),
    "Medial Meniscus": (
        "Pathology of the medial meniscus, including tear, "
        "degeneration, maceration, extrusion, or prior resection."
    ),
    "Lateral Meniscus": (
        "Pathology of the lateral meniscus, including tear, "
        "degeneration, maceration, extrusion, or prior resection."
    ),
    "Medial OA": (
        "Osteoarthritis or degenerative arthrosis involving the "
        "medial tibiofemoral compartment."
    ),
    "Lateral OA": (
        "Osteoarthritis or degenerative arthrosis involving the "
        "lateral tibiofemoral compartment."
    ),
    "PF OA": (
        "Patellofemoral osteoarthritis or degenerative arthrosis."
    ),
    "Effusion": (
        "Knee joint effusion or increased intra-articular fluid."
    ),
    "Synovitis": (
        "Synovitis, synovial hypertrophy, or inflammatory synovial "
        "proliferation. Do not infer it from effusion alone."
    ),
    "Baker's": (
        "Baker or popliteal cyst."
    ),
    "Contusion": (
        "Bone contusion, bone bruise, or traumatic marrow edema."
    ),
    "Fracture": (
        "Acute, healing, stress, insufficiency, or osteochondral "
        "fracture."
    ),
}

target_instructions = "\n".join(
    f'- "{target}": {description}'
    for target, description
    in TARGET_DESCRIPTIONS.items()
)

system_prompt = f"""
You extract structured findings from knee MRI radiology reports.

Reports may be written in English, Greek, Cyrillic-script languages,
or other European languages. Interpret the original language directly.

Evaluate exactly these targets:
{target_instructions}

For every target, assign exactly one status:

- present: the report states that the abnormality exists.
- absent: the report explicitly denies it or describes the structure
  as normal or intact.
- uncertain: the report uses uncertain language such as possible,
  suspected, cannot exclude, equivocal, or questionable.
- not_mentioned: the report provides no usable statement about it.

Rules:
- Do not treat missing information as absent.
- Respect negation and uncertainty.
- Do not infer synovitis from effusion alone.
- Do not infer fracture from nonspecific marrow edema.
- Do not infer osteoarthritis from an isolated focal cartilage lesion
  unless degenerative arthrosis or osteoarthritis is stated.
- Prior surgery or resection counts as present pathology for the
  affected structure.
- severity must be one of: none, mild, moderate, severe, unspecified.
- Use severity "none" for status absent or not_mentioned.
- Use "unspecified" when pathology is present but no grade is stated.
- negated is true only when explicit negation supports status absent.
- Do not quote, repeat, translate, or summarize the report.
- Return one JSON object and no other text.
- The JSON must contain exactly the twelve target names.
- Each target value must contain exactly status, severity, and negated.
""".strip()

development_row_indices = gold_indices[
    development_positions
]

non_latin_development_indices = [
    row_index
    for row_index in development_row_indices
    if dominant_script.loc[row_index] != "LATIN"
]

if non_latin_development_indices:
    example_index = non_latin_development_indices[0]
else:
    example_index = development_row_indices[0]

example_report = reports.loc[example_index]
example_script = dominant_script.loc[example_index]

messages = [
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": (
            "Extract the twelve target states from this report:\n\n"
            + example_report
        ),
    },
]

model_inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(report_model.device)

torch.cuda.synchronize()
generation_start = time.perf_counter()

with torch.inference_mode():
    generated = report_model.generate(
        **model_inputs,
        max_new_tokens=1000,
        do_sample=False,
        use_cache=True,
    )

torch.cuda.synchronize()
generation_seconds = (
    time.perf_counter() - generation_start
)

prompt_length = model_inputs["input_ids"].shape[1]

response_text = tokenizer.decode(
    generated[0, prompt_length:],
    skip_special_tokens=True,
).strip()

# Parse only the outermost JSON object.
json_start = response_text.find("{")
json_end = response_text.rfind("}")

if json_start < 0 or json_end <= json_start:
    debug_path = Path(
        "/kaggle/working/report_extraction_debug.txt"
    )
    debug_path.write_text(
        response_text,
        encoding="utf-8",
    )
    raise RuntimeError(
        "The model did not return a JSON object. "
        f"Raw response saved privately to {debug_path}"
    )

parsed = json.loads(
    response_text[
        json_start:json_end + 1
    ]
)

if set(parsed) != set(TARGETS):
    raise RuntimeError(
        "Returned target keys do not match the required targets. "
        f"Returned keys: {sorted(parsed)}"
    )

rows = []

for target in TARGETS:
    result = parsed[target]

    if set(result) != {
        "status",
        "severity",
        "negated",
    }:
        raise RuntimeError(
            f"Invalid fields for {target}: {result}"
        )

    if result["status"] not in STATUS_VALUES:
        raise RuntimeError(
            f"Invalid status for {target}: {result['status']}"
        )

    if result["severity"] not in SEVERITY_VALUES:
        raise RuntimeError(
            f"Invalid severity for {target}: {result['severity']}"
        )

    if not isinstance(result["negated"], bool):
        raise RuntimeError(
            f"Invalid negated value for {target}"
        )

    rows.append(
        {
            "target": target,
            "status": result["status"],
            "severity": result["severity"],
            "negated": result["negated"],
            "gold_label": int(
                train.loc[
                    example_index,
                    target,
                ]
            ),
        }
    )

result_table = pd.DataFrame(rows)

print("Development example script:", example_script)
print("Prompt tokens:", prompt_length)
print(
    "Generated tokens:",
    generated.shape[1] - prompt_length,
)
print(
    "Generation time (seconds):",
    round(generation_seconds, 2),
)
print("Schema validation passed.")

display(result_table)

In [ ]:
def validate_and_normalize_extraction(parsed_result):
    normalized = {}
    corrections = []

    if set(parsed_result) != set(TARGETS):
        raise ValueError(
            "Target keys do not match the required target set"
        )

    for target in TARGETS:
        result = parsed_result[target]

        if set(result) != {
            "status",
            "severity",
            "negated",
        }:
            raise ValueError(
                f"Invalid fields for {target}: {result}"
            )

        status = result["status"]
        severity = result["severity"]
        negated = result["negated"]

        if status not in STATUS_VALUES:
            raise ValueError(
                f"Invalid status for {target}: {status}"
            )

        if severity not in SEVERITY_VALUES:
            raise ValueError(
                f"Invalid severity for {target}: {severity}"
            )

        if not isinstance(negated, bool):
            raise ValueError(
                f"Invalid negated value for {target}"
            )

        # Negation can only support an absent classification.
        if negated and status != "absent":
            corrections.append(
                {
                    "target": target,
                    "field": "negated",
                    "original": negated,
                    "normalized": False,
                    "reason": (
                        "negated=True is incompatible "
                        f"with status={status}"
                    ),
                }
            )
            negated = False

        # Absent and unmentioned findings cannot carry severity.
        if (
            status in {
                "absent",
                "not_mentioned",
            }
            and severity != "none"
        ):
            corrections.append(
                {
                    "target": target,
                    "field": "severity",
                    "original": severity,
                    "normalized": "none",
                    "reason": (
                        f"severity is incompatible with status={status}"
                    ),
                }
            )
            severity = "none"

        # Present pathology without a grade has unspecified severity.
        if status == "present" and severity == "none":
            corrections.append(
                {
                    "target": target,
                    "field": "severity",
                    "original": severity,
                    "normalized": "unspecified",
                    "reason": (
                        "present pathology cannot have severity=none"
                    ),
                }
            )
            severity = "unspecified"

        normalized[target] = {
            "status": status,
            "severity": severity,
            "negated": negated,
        }

    return normalized, corrections


normalized_example, example_corrections = (
    validate_and_normalize_extraction(parsed)
)

normalized_rows = []

for target in TARGETS:
    result = normalized_example[target]

    normalized_rows.append(
        {
            "target": target,
            "status": result["status"],
            "severity": result["severity"],
            "negated": result["negated"],
            "gold_label": int(
                train.loc[
                    example_index,
                    target,
                ]
            ),
        }
    )

print(
    "Semantic corrections:",
    len(example_corrections),
)

display(pd.DataFrame(example_corrections))

print("\nNormalized extraction:")
display(pd.DataFrame(normalized_rows))

In [ ]:
import json
import time
from pathlib import Path

DEVELOPMENT_BATCH_SIZE = 2
MAX_NEW_TOKENS = 700

development_output_path = Path(
    "/kaggle/working/development_report_extractions.jsonl"
)

debug_directory = Path(
    "/kaggle/working/report_extraction_debug"
)
debug_directory.mkdir(
    parents=True,
    exist_ok=True,
)

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_report_messages(report_text):
    return [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": (
                "Extract the twelve target states "
                "from this report:\n\n"
                + report_text
            ),
        },
    ]

def parse_report_response(response_text):
    json_start = response_text.find("{")
    json_end = response_text.rfind("}")

    if json_start < 0 or json_end <= json_start:
        raise ValueError(
            "Response does not contain a JSON object"
        )

    parsed_result = json.loads(
        response_text[
            json_start:json_end + 1
        ]
    )

    return validate_and_normalize_extraction(
        parsed_result
    )

development_records = []
development_start = time.perf_counter()

for batch_start in range(
    0,
    len(development_row_indices),
    DEVELOPMENT_BATCH_SIZE,
):
    batch_indices = development_row_indices[
        batch_start:
        batch_start + DEVELOPMENT_BATCH_SIZE
    ]

    conversations = [
        build_report_messages(
            reports.loc[row_index]
        )
        for row_index in batch_indices
    ]

    batch_inputs = tokenizer.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        padding=True,
        return_dict=True,
        return_tensors="pt",
    ).to(report_model.device)

    input_width = batch_inputs["input_ids"].shape[1]

    batch_generation_start = time.perf_counter()

    with torch.inference_mode():
        batch_generated = report_model.generate(
            **batch_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            use_cache=True,
        )

    torch.cuda.synchronize()

    batch_seconds = (
        time.perf_counter()
        - batch_generation_start
    )

    for local_index, row_index in enumerate(
        batch_indices
    ):
        response_text = tokenizer.decode(
            batch_generated[
                local_index,
                input_width:,
            ],
            skip_special_tokens=True,
        ).strip()

        record = {
            "study_uid": str(
                train.loc[
                    row_index,
                    "StudyInstanceUID",
                ]
            ),
            "script": str(
                dominant_script.loc[
                    row_index
                ]
            ),
            "prompt_tokens": int(
                batch_inputs[
                    "attention_mask"
                ][local_index].sum()
            ),
            "generated_tokens": int(
                batch_generated.shape[1]
                - input_width
            ),
            "predictions": None,
            "corrections": [],
            "error": None,
        }

        try:
            normalized, corrections = (
                parse_report_response(
                    response_text
                )
            )

            record["predictions"] = normalized
            record["corrections"] = corrections

        except Exception as error:
            record["error"] = (
                f"{type(error).__name__}: {error}"
            )

            debug_path = (
                debug_directory
                / (
                    "development_"
                    f"{batch_start + local_index:03d}.txt"
                )
            )

            debug_path.write_text(
                response_text,
                encoding="utf-8",
            )

        development_records.append(record)

    with development_output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for record in development_records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    completed = min(
        batch_start + len(batch_indices),
        len(development_row_indices),
    )

    print(
        f"Processed {completed:02d}/"
        f"{len(development_row_indices):02d} "
        f"in {batch_seconds:.1f} seconds"
    )

    del batch_inputs
    del batch_generated
    torch.cuda.empty_cache()

development_seconds = (
    time.perf_counter()
    - development_start
)

successful_records = [
    record
    for record in development_records
    if record["error"] is None
]

failed_records = [
    record
    for record in development_records
    if record["error"] is not None
]

correction_count = sum(
    len(record["corrections"])
    for record in successful_records
)

status_counts = {
    status: 0
    for status in sorted(STATUS_VALUES)
}

for record in successful_records:
    for target in TARGETS:
        status = record["predictions"][target]["status"]
        status_counts[status] += 1

print()
print(
    "Development extraction time (minutes):",
    round(development_seconds / 60, 2),
)
print("Successful reports:", len(successful_records))
print("Failed reports:", len(failed_records))
print("Semantic corrections:", correction_count)
print("Checkpoint:", development_output_path)
print("Status counts:", status_counts)

In [ ]:
from collections import Counter

error_summary = Counter(
    record["error"]
    for record in failed_records
)

failure_rows = []

for position, record in enumerate(
    development_records
):
    if record["error"] is None:
        continue

    debug_path = (
        debug_directory
        / f"development_{position:03d}.txt"
    )

    failure_rows.append(
        {
            "position": position,
            "script": record["script"],
            "prompt_tokens": record["prompt_tokens"],
            "generated_tokens": record["generated_tokens"],
            "response_characters": (
                debug_path.stat().st_size
                if debug_path.exists()
                else None
            ),
            "error": record["error"],
        }
    )

correction_rows = []

for record in successful_records:
    for correction in record["corrections"]:
        correction_rows.append(
            {
                "target": correction["target"],
                "field": correction["field"],
                "reason": correction["reason"],
            }
        )

correction_summary = (
    pd.DataFrame(correction_rows)
    .groupby(
        [
            "target",
            "field",
            "reason",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        "count",
        ascending=False,
    )
    if correction_rows
    else pd.DataFrame(
        columns=[
            "target",
            "field",
            "reason",
            "count",
        ]
    )
)

print("Error summary:")

for error, count in error_summary.items():
    print(f"- {count} × {error}")

print("\nFailed response diagnostics:")
display(pd.DataFrame(failure_rows))

print("\nSemantic corrections by target and field:")
display(correction_summary)

In [ ]:
SEVERITY_VALUES = {
    "none",
    "mild",
    "mild_to_moderate",
    "moderate",
    "moderate_to_severe",
    "severe",
    "unspecified",
}

SEVERITY_ALIASES = {
    "mild_to_moderate": "mild_to_moderate",
    "mild_moderate": "mild_to_moderate",
    "moderate_to_severe": "moderate_to_severe",
    "moderate_severe": "moderate_to_severe",
}

def normalize_severity_value(value):
    normalized = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    return SEVERITY_ALIASES.get(
        normalized,
        normalized,
    )

def validate_and_normalize_extraction(parsed_result):
    normalized = {}
    corrections = []

    if set(parsed_result) != set(TARGETS):
        raise ValueError(
            "Target keys do not match the required target set"
        )

    for target in TARGETS:
        result = parsed_result[target]

        if set(result) != {
            "status",
            "severity",
            "negated",
        }:
            raise ValueError(
                f"Invalid fields for {target}: {result}"
            )

        status = result["status"]
        original_severity = result["severity"]
        severity = normalize_severity_value(
            original_severity
        )
        negated = result["negated"]

        if status not in STATUS_VALUES:
            raise ValueError(
                f"Invalid status for {target}: {status}"
            )

        if severity not in SEVERITY_VALUES:
            raise ValueError(
                f"Invalid severity for {target}: "
                f"{original_severity}"
            )

        if severity != original_severity:
            corrections.append(
                {
                    "target": target,
                    "field": "severity",
                    "original": original_severity,
                    "normalized": severity,
                    "reason": "normalized severity spelling",
                }
            )

        if not isinstance(negated, bool):
            raise ValueError(
                f"Invalid negated value for {target}"
            )

        if negated and status != "absent":
            corrections.append(
                {
                    "target": target,
                    "field": "negated",
                    "original": True,
                    "normalized": False,
                    "reason": (
                        "negated=True is incompatible "
                        f"with status={status}"
                    ),
                }
            )
            negated = False

        if (
            status in {
                "absent",
                "not_mentioned",
            }
            and severity != "none"
        ):
            corrections.append(
                {
                    "target": target,
                    "field": "severity",
                    "original": severity,
                    "normalized": "none",
                    "reason": (
                        f"severity is incompatible with status={status}"
                    ),
                }
            )
            severity = "none"

        if status == "present" and severity == "none":
            corrections.append(
                {
                    "target": target,
                    "field": "severity",
                    "original": severity,
                    "normalized": "unspecified",
                    "reason": (
                        "present pathology cannot have severity=none"
                    ),
                }
            )
            severity = "unspecified"

        normalized[target] = {
            "status": status,
            "severity": severity,
            "negated": negated,
        }

    return normalized, corrections


# Recover the two previously rejected responses without inference.
for position, record in enumerate(
    development_records
):
    if record["error"] is None:
        continue

    debug_path = (
        debug_directory
        / f"development_{position:03d}.txt"
    )

    response_text = debug_path.read_text(
        encoding="utf-8"
    )

    normalized, corrections = (
        parse_report_response(response_text)
    )

    record["predictions"] = normalized
    record["corrections"] = corrections
    record["error"] = None

successful_records = [
    record
    for record in development_records
    if record["error"] is None
]

failed_records = [
    record
    for record in development_records
    if record["error"] is not None
]

with development_output_path.open(
    "w",
    encoding="utf-8",
) as file:
    for record in development_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

uid_to_row_index = dict(
    zip(
        train["StudyInstanceUID"].astype(str),
        train.index,
    )
)

contradiction_rows = []

for record in successful_records:
    row_index = uid_to_row_index[
        record["study_uid"]
    ]

    for correction in record["corrections"]:
        if (
            correction["field"] == "negated"
            and correction["original"] is True
        ):
            target = correction["target"]

            contradiction_rows.append(
                {
                    "target": target,
                    "gold_label": int(
                        train.loc[
                            row_index,
                            target,
                        ]
                    ),
                }
            )

contradiction_frame = pd.DataFrame(
    contradiction_rows
)

contradiction_summary = (
    contradiction_frame
    .groupby("target")
    .agg(
        cases=("gold_label", "size"),
        gold_negative=(
            "gold_label",
            lambda values: int((values == 0).sum()),
        ),
        gold_positive=(
            "gold_label",
            lambda values: int((values == 1).sum()),
        ),
    )
    .sort_values(
        "cases",
        ascending=False,
    )
)

print("Successful reports:", len(successful_records))
print("Failed reports:", len(failed_records))
print(
    "Recovered reports:",
    2 - len(failed_records),
)

print(
    "\nGold labels for not_mentioned + negated=True contradictions:"
)
display(contradiction_summary)

In [ ]:
from sklearn.metrics import roc_auc_score

prediction_rows = []

neutral_scores = {
    "present": 1.0,
    "absent": 0.0,
    "uncertain": 0.5,
    "not_mentioned": 0.5,
}

for record in successful_records:
    row_index = uid_to_row_index[
        record["study_uid"]
    ]

    for target in TARGETS:
        result = record["predictions"][target]
        status = result["status"]
        gold_label = int(
            train.loc[
                row_index,
                target,
            ]
        )

        is_resolved = status in {
            "present",
            "absent",
        }

        resolved_prediction = (
            int(status == "present")
            if is_resolved
            else None
        )

        prediction_rows.append(
            {
                "target": target,
                "script": record["script"],
                "status": status,
                "gold_label": gold_label,
                "is_resolved": is_resolved,
                "resolved_prediction": resolved_prediction,
                "neutral_score": neutral_scores[status],
            }
        )

development_predictions = pd.DataFrame(
    prediction_rows
)

target_metric_rows = []

for target in TARGETS:
    target_frame = development_predictions[
        development_predictions["target"] == target
    ]

    resolved = target_frame[
        target_frame["is_resolved"]
    ].copy()

    true_positive = int(
        (
            (resolved["gold_label"] == 1)
            & (resolved["resolved_prediction"] == 1)
        ).sum()
    )

    true_negative = int(
        (
            (resolved["gold_label"] == 0)
            & (resolved["resolved_prediction"] == 0)
        ).sum()
    )

    false_positive = int(
        (
            (resolved["gold_label"] == 0)
            & (resolved["resolved_prediction"] == 1)
        ).sum()
    )

    false_negative = int(
        (
            (resolved["gold_label"] == 1)
            & (resolved["resolved_prediction"] == 0)
        ).sum()
    )

    resolved_count = len(resolved)

    resolved_accuracy = (
        (true_positive + true_negative)
        / resolved_count
        if resolved_count
        else float("nan")
    )

    sensitivity = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative
        else float("nan")
    )

    specificity = (
        true_negative
        / (true_negative + false_positive)
        if true_negative + false_positive
        else float("nan")
    )

    neutral_auc = roc_auc_score(
        target_frame["gold_label"],
        target_frame["neutral_score"],
    )

    target_metric_rows.append(
        {
            "target": target,
            "resolved": resolved_count,
            "coverage": resolved_count / len(target_frame),
            "resolved_accuracy": resolved_accuracy,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "neutral_unknown_auc": neutral_auc,
            "present": int(
                (target_frame["status"] == "present").sum()
            ),
            "absent": int(
                (target_frame["status"] == "absent").sum()
            ),
            "uncertain": int(
                (target_frame["status"] == "uncertain").sum()
            ),
            "not_mentioned": int(
                (
                    target_frame["status"]
                    == "not_mentioned"
                ).sum()
            ),
        }
    )

development_metrics = (
    pd.DataFrame(target_metric_rows)
    .set_index("target")
)

resolved_all = development_predictions[
    development_predictions["is_resolved"]
]

overall_resolved_accuracy = (
    resolved_all["gold_label"]
    == resolved_all["resolved_prediction"]
).mean()

script_metrics = (
    development_predictions
    .groupby("script")
    .apply(
        lambda frame: pd.Series(
            {
                "decisions": len(frame),
                "resolved": int(
                    frame["is_resolved"].sum()
                ),
                "coverage": frame["is_resolved"].mean(),
                "resolved_accuracy": (
                    (
                        frame.loc[
                            frame["is_resolved"],
                            "gold_label",
                        ]
                        == frame.loc[
                            frame["is_resolved"],
                            "resolved_prediction",
                        ]
                    ).mean()
                    if frame["is_resolved"].any()
                    else float("nan")
                ),
            }
        ),
        include_groups=False,
    )
)

print(
    "Overall resolved coverage:",
    round(
        development_predictions["is_resolved"].mean(),
        4,
    ),
)
print(
    "Overall resolved accuracy:",
    round(
        overall_resolved_accuracy,
        4,
    ),
)
print(
    "Macro neutral-unknown AUC:",
    round(
        development_metrics[
            "neutral_unknown_auc"
        ].mean(),
        4,
    ),
)

print("\nPer-target development metrics:")
display(
    development_metrics.style.format(
        {
            "coverage": "{:.3f}",
            "resolved_accuracy": "{:.3f}",
            "sensitivity": "{:.3f}",
            "specificity": "{:.3f}",
            "neutral_unknown_auc": "{:.3f}",
        }
    )
)

print("\nMetrics by writing system:")
display(
    script_metrics.style.format(
        {
            "coverage": "{:.3f}",
            "resolved_accuracy": "{:.3f}",
        }
    )
)

In [ ]:
supervision_metric_rows = []

for target in TARGETS:
    frame = development_predictions[
        development_predictions["target"] == target
    ]

    gold_positive = frame["gold_label"] == 1
    gold_negative = frame["gold_label"] == 0
    predicted_positive = frame["status"] == "present"
    predicted_negative = frame["status"] == "absent"

    true_positive = int(
        (gold_positive & predicted_positive).sum()
    )
    false_positive = int(
        (gold_negative & predicted_positive).sum()
    )
    true_negative = int(
        (gold_negative & predicted_negative).sum()
    )
    false_negative = int(
        (gold_positive & predicted_negative).sum()
    )

    positive_precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive
        else float("nan")
    )

    negative_precision = (
        true_negative
        / (true_negative + false_negative)
        if true_negative + false_negative
        else float("nan")
    )

    positive_gold_coverage = (
        true_positive
        / gold_positive.sum()
    )

    negative_gold_coverage = (
        true_negative
        / gold_negative.sum()
    )

    supervision_metric_rows.append(
        {
            "target": target,
            "gold_positive": int(gold_positive.sum()),
            "gold_negative": int(gold_negative.sum()),
            "pseudo_positive": int(predicted_positive.sum()),
            "pseudo_negative": int(predicted_negative.sum()),
            "positive_precision": positive_precision,
            "negative_precision": negative_precision,
            "positive_gold_coverage": positive_gold_coverage,
            "negative_gold_coverage": negative_gold_coverage,
            "unknown_rate": float(
                frame["status"]
                .isin(
                    {
                        "uncertain",
                        "not_mentioned",
                    }
                )
                .mean()
            ),
        }
    )

supervision_metrics = (
    pd.DataFrame(supervision_metric_rows)
    .set_index("target")
)

generated_positive_count = int(
    (
        development_predictions["status"]
        == "present"
    ).sum()
)

correct_generated_positive_count = int(
    (
        (
            development_predictions["status"]
            == "present"
        )
        & (
            development_predictions["gold_label"]
            == 1
        )
    ).sum()
)

generated_negative_count = int(
    (
        development_predictions["status"]
        == "absent"
    ).sum()
)

correct_generated_negative_count = int(
    (
        (
            development_predictions["status"]
            == "absent"
        )
        & (
            development_predictions["gold_label"]
            == 0
        )
    ).sum()
)

micro_positive_precision = (
    correct_generated_positive_count
    / generated_positive_count
)

micro_negative_precision = (
    correct_generated_negative_count
    / generated_negative_count
)

print(
    "Micro pseudo-positive precision:",
    round(micro_positive_precision, 4),
)
print(
    "Micro pseudo-negative precision:",
    round(micro_negative_precision, 4),
)
print(
    "Generated positive labels:",
    generated_positive_count,
)
print(
    "Generated negative labels:",
    generated_negative_count,
)

display(
    supervision_metrics.style.format(
        {
            "positive_precision": "{:.3f}",
            "negative_precision": "{:.3f}",
            "positive_gold_coverage": "{:.3f}",
            "negative_gold_coverage": "{:.3f}",
            "unknown_rate": "{:.3f}",
        }
    )
)

In [ ]:
severity_rows = []

for record in successful_records:
    row_index = uid_to_row_index[
        record["study_uid"]
    ]

    for target in TARGETS:
        result = record["predictions"][target]

        if result["status"] != "present":
            continue

        severity_rows.append(
            {
                "target": target,
                "severity": result["severity"],
                "gold_label": int(
                    train.loc[
                        row_index,
                        target,
                    ]
                ),
            }
        )

present_by_severity = pd.DataFrame(
    severity_rows
)

overall_severity_precision = (
    present_by_severity
    .groupby("severity")
    .agg(
        pseudo_positive=("gold_label", "size"),
        true_positive=("gold_label", "sum"),
    )
)

overall_severity_precision[
    "precision"
] = (
    overall_severity_precision["true_positive"]
    / overall_severity_precision["pseudo_positive"]
)

target_severity_precision = (
    present_by_severity
    .groupby(
        [
            "target",
            "severity",
        ]
    )
    .agg(
        pseudo_positive=("gold_label", "size"),
        true_positive=("gold_label", "sum"),
    )
)

target_severity_precision[
    "precision"
] = (
    target_severity_precision["true_positive"]
    / target_severity_precision["pseudo_positive"]
)

print("Pseudo-positive precision by severity:")
display(
    overall_severity_precision.style.format(
        {
            "precision": "{:.3f}",
        }
    )
)

print("\nPseudo-positive precision by target and severity:")
display(
    target_severity_precision.style.format(
        {
            "precision": "{:.3f}",
        }
    )
)

In [ ]:
TARGET_DESCRIPTIONS_V2 = {
    "ACL": (
        "Explicitly diagnosed ACL tear, rupture, sprain, "
        "degeneration, or postoperative ACL reconstruction."
    ),
    "MCL": (
        "Explicitly diagnosed MCL tear, rupture, sprain, "
        "or chronic MCL injury."
    ),
    "Medial Meniscus": (
        "Explicitly diagnosed medial meniscal tear, degeneration, "
        "maceration, extrusion, or prior partial meniscectomy."
    ),
    "Lateral Meniscus": (
        "Explicitly diagnosed lateral meniscal tear, degeneration, "
        "maceration, extrusion, or prior partial meniscectomy."
    ),
    "Medial OA": (
        "Explicit medial-compartment osteoarthritis, arthrosis, "
        "or compartment-level degenerative joint disease."
    ),
    "Lateral OA": (
        "Explicit lateral-compartment osteoarthritis, arthrosis, "
        "or compartment-level degenerative joint disease."
    ),
    "PF OA": (
        "Explicit patellofemoral osteoarthritis, arthrosis, "
        "or patellofemoral degenerative joint disease."
    ),
    "Effusion": (
        "Explicit knee joint effusion. Trace physiologic fluid "
        "without a diagnosis of effusion is uncertain."
    ),
    "Synovitis": (
        "Explicit synovitis or synovial inflammatory proliferation."
    ),
    "Baker's": (
        "Explicit Baker cyst or popliteal cyst."
    ),
    "Contusion": (
        "Explicit bone contusion or bone bruise. Nonspecific marrow "
        "edema is not sufficient unless called traumatic contusion."
    ),
    "Fracture": (
        "Explicit acute, healing, stress, insufficiency, "
        "osteochondral, or avulsion fracture."
    ),
}

target_instructions_v2 = "\n".join(
    f'- "{target}": {description}'
    for target, description
    in TARGET_DESCRIPTIONS_V2.items()
)

system_prompt_v2 = f"""
You conservatively extract structured findings from knee MRI
radiology reports.

Reports may be written in English, Greek, Cyrillic-script languages,
or other European languages. Interpret the original language directly.

Evaluate exactly these targets:
{target_instructions_v2}

For every target, assign exactly one status:

- present: the report explicitly diagnoses the abnormality or uses
  an unambiguous synonymous diagnosis.
- absent: the report explicitly denies the abnormality or explicitly
  describes the relevant structure as normal or intact.
- uncertain: the report is equivocal, uses possible/suspected
  language, or supplies only indirect evidence from which the
  abnormality might be inferred.
- not_mentioned: no usable statement is made about the target.

Conservative rules:
- Do not classify an abnormality as present from indirect evidence.
- Do not treat missing information as absent.
- Isolated abnormal signal is insufficient unless the report
  diagnoses the named ligament or meniscal pathology.
- An isolated focal cartilage lesion is insufficient for OA.
- Nonspecific marrow edema is insufficient for contusion.
- Effusion alone is insufficient for synovitis.
- Marrow edema alone is insufficient for fracture.
- Preserve uncertainty rather than guessing.
- Prior surgery or resection counts as present only for the structure
  explicitly named as operated on.
- severity must be one of: none, mild, mild_to_moderate, moderate,
  moderate_to_severe, severe, unspecified.
- Use severity none for absent and not_mentioned.
- Use unspecified when explicit pathology is present without a grade.
- negated may be true only when status is absent.
- Do not quote, repeat, translate, or summarize the report.
- Return one JSON object and no other text.
- Include exactly the twelve target names.
- Each target must contain exactly status, severity, and negated.
""".strip()

contusion_false_positive_records = []

for record in successful_records:
    row_index = uid_to_row_index[
        record["study_uid"]
    ]

    if (
        record["predictions"]["Contusion"]["status"]
        == "present"
        and int(train.loc[row_index, "Contusion"]) == 0
    ):
        contusion_false_positive_records.append(
            (record, row_index)
        )

if not contusion_false_positive_records:
    raise RuntimeError(
        "No development contusion false positive found"
    )

v1_record, v2_example_index = (
    contusion_false_positive_records[0]
)

v2_messages = [
    {
        "role": "system",
        "content": system_prompt_v2,
    },
    {
        "role": "user",
        "content": (
            "Extract the twelve target states from this report:\n\n"
            + reports.loc[v2_example_index]
        ),
    },
]

v2_inputs = tokenizer.apply_chat_template(
    v2_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(report_model.device)

v2_start = time.perf_counter()

with torch.inference_mode():
    v2_generated = report_model.generate(
        **v2_inputs,
        max_new_tokens=700,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        use_cache=True,
    )

torch.cuda.synchronize()
v2_seconds = time.perf_counter() - v2_start

v2_prompt_width = v2_inputs["input_ids"].shape[1]

v2_response = tokenizer.decode(
    v2_generated[0, v2_prompt_width:],
    skip_special_tokens=True,
).strip()

v2_normalized, v2_corrections = (
    parse_report_response(v2_response)
)

comparison_rows = []

for target in TARGETS:
    comparison_rows.append(
        {
            "target": target,
            "v1_status": (
                v1_record["predictions"][target]["status"]
            ),
            "v2_status": (
                v2_normalized[target]["status"]
            ),
            "gold_label": int(
                train.loc[
                    v2_example_index,
                    target,
                ]
            ),
        }
    )

print(
    "Development example script:",
    dominant_script.loc[v2_example_index],
)
print(
    "Generation time (seconds):",
    round(v2_seconds, 2),
)
print(
    "V2 semantic corrections:",
    len(v2_corrections),
)
print(
    "Contusion V1:",
    v1_record["predictions"]["Contusion"]["status"],
)
print(
    "Contusion V2:",
    v2_normalized["Contusion"]["status"],
)
print(
    "Contusion gold:",
    int(train.loc[v2_example_index, "Contusion"]),
)

display(pd.DataFrame(comparison_rows))

In [ ]:
V2_BATCH_SIZE = 2
V2_MAX_NEW_TOKENS = 700

v2_output_path = Path(
    "/kaggle/working/development_report_extractions_v2.jsonl"
)

def build_v2_messages(report_text):
    return [
        {
            "role": "system",
            "content": system_prompt_v2,
        },
        {
            "role": "user",
            "content": (
                "Extract the twelve target states "
                "from this report:\n\n"
                + report_text
            ),
        },
    ]

development_records_v2 = []
v2_development_start = time.perf_counter()

for batch_start in range(
    0,
    len(development_row_indices),
    V2_BATCH_SIZE,
):
    batch_indices = development_row_indices[
        batch_start:
        batch_start + V2_BATCH_SIZE
    ]

    conversations = [
        build_v2_messages(
            reports.loc[row_index]
        )
        for row_index in batch_indices
    ]

    batch_inputs = tokenizer.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        padding=True,
        return_dict=True,
        return_tensors="pt",
    ).to(report_model.device)

    input_width = batch_inputs["input_ids"].shape[1]

    batch_start_time = time.perf_counter()

    with torch.inference_mode():
        batch_generated = report_model.generate(
            **batch_inputs,
            max_new_tokens=V2_MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            use_cache=True,
        )

    torch.cuda.synchronize()

    batch_seconds = (
        time.perf_counter()
        - batch_start_time
    )

    for local_index, row_index in enumerate(
        batch_indices
    ):
        response_text = tokenizer.decode(
            batch_generated[
                local_index,
                input_width:,
            ],
            skip_special_tokens=True,
        ).strip()

        record = {
            "study_uid": str(
                train.loc[
                    row_index,
                    "StudyInstanceUID",
                ]
            ),
            "script": str(
                dominant_script.loc[
                    row_index
                ]
            ),
            "predictions": None,
            "corrections": [],
            "error": None,
        }

        try:
            normalized, corrections = (
                parse_report_response(
                    response_text
                )
            )

            record["predictions"] = normalized
            record["corrections"] = corrections

        except Exception as error:
            record["error"] = (
                f"{type(error).__name__}: {error}"
            )

            debug_path = (
                debug_directory
                / (
                    "development_v2_"
                    f"{batch_start + local_index:03d}.txt"
                )
            )

            debug_path.write_text(
                response_text,
                encoding="utf-8",
            )

        development_records_v2.append(record)

    with v2_output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for record in development_records_v2:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    completed = min(
        batch_start + len(batch_indices),
        len(development_row_indices),
    )

    print(
        f"Processed {completed:02d}/"
        f"{len(development_row_indices):02d} "
        f"in {batch_seconds:.1f} seconds"
    )

    del batch_inputs
    del batch_generated
    torch.cuda.empty_cache()

v2_total_seconds = (
    time.perf_counter()
    - v2_development_start
)

successful_records_v2 = [
    record
    for record in development_records_v2
    if record["error"] is None
]

failed_records_v2 = [
    record
    for record in development_records_v2
    if record["error"] is not None
]

v2_correction_count = sum(
    len(record["corrections"])
    for record in successful_records_v2
)

print()
print(
    "V2 extraction time (minutes):",
    round(v2_total_seconds / 60, 2),
)
print(
    "Successful reports:",
    len(successful_records_v2),
)
print(
    "Failed reports:",
    len(failed_records_v2),
)
print(
    "Semantic corrections:",
    v2_correction_count,
)
print("Checkpoint:", v2_output_path)

In [ ]:
def records_to_prediction_frame(records):
    rows = []

    for record in records:
        row_index = uid_to_row_index[
            record["study_uid"]
        ]

        for target in TARGETS:
            result = record["predictions"][target]
            status = result["status"]

            rows.append(
                {
                    "study_uid": record["study_uid"],
                    "target": target,
                    "script": record["script"],
                    "status": status,
                    "severity": result["severity"],
                    "gold_label": int(
                        train.loc[
                            row_index,
                            target,
                        ]
                    ),
                    "is_resolved": status in {
                        "present",
                        "absent",
                    },
                    "neutral_score": neutral_scores[status],
                }
            )

    return pd.DataFrame(rows)


def calculate_prompt_metrics(prediction_frame):
    target_rows = []

    for target in TARGETS:
        frame = prediction_frame[
            prediction_frame["target"] == target
        ]

        gold_positive = frame["gold_label"] == 1
        gold_negative = frame["gold_label"] == 0
        predicted_positive = frame["status"] == "present"
        predicted_negative = frame["status"] == "absent"

        true_positive = int(
            (gold_positive & predicted_positive).sum()
        )
        false_positive = int(
            (gold_negative & predicted_positive).sum()
        )
        true_negative = int(
            (gold_negative & predicted_negative).sum()
        )
        false_negative = int(
            (gold_positive & predicted_negative).sum()
        )

        positive_precision = (
            true_positive
            / (true_positive + false_positive)
            if true_positive + false_positive
            else float("nan")
        )

        negative_precision = (
            true_negative
            / (true_negative + false_negative)
            if true_negative + false_negative
            else float("nan")
        )

        target_rows.append(
            {
                "target": target,
                "positive_precision": positive_precision,
                "negative_precision": negative_precision,
                "positive_gold_coverage": (
                    true_positive / gold_positive.sum()
                ),
                "negative_gold_coverage": (
                    true_negative / gold_negative.sum()
                ),
                "resolved_coverage": float(
                    (
                        predicted_positive
                        | predicted_negative
                    ).mean()
                ),
                "unknown_rate": float(
                    (
                        ~predicted_positive
                        & ~predicted_negative
                    ).mean()
                ),
                "neutral_unknown_auc": roc_auc_score(
                    frame["gold_label"],
                    frame["neutral_score"],
                ),
            }
        )

    target_metrics = (
        pd.DataFrame(target_rows)
        .set_index("target")
    )

    predicted_positive = (
        prediction_frame["status"] == "present"
    )

    predicted_negative = (
        prediction_frame["status"] == "absent"
    )

    micro_positive_precision = (
        prediction_frame.loc[
            predicted_positive,
            "gold_label",
        ].mean()
    )

    micro_negative_precision = (
        1
        - prediction_frame.loc[
            predicted_negative,
            "gold_label",
        ].mean()
    )

    summary = {
        "micro_positive_precision":
            micro_positive_precision,
        "micro_negative_precision":
            micro_negative_precision,
        "resolved_coverage":
            prediction_frame["is_resolved"].mean(),
        "macro_neutral_unknown_auc":
            target_metrics[
                "neutral_unknown_auc"
            ].mean(),
        "pseudo_positive_count":
            int(predicted_positive.sum()),
        "pseudo_negative_count":
            int(predicted_negative.sum()),
    }

    return summary, target_metrics


predictions_v1 = records_to_prediction_frame(
    successful_records
)

predictions_v2 = records_to_prediction_frame(
    successful_records_v2
)

summary_v1, metrics_v1 = (
    calculate_prompt_metrics(predictions_v1)
)

summary_v2, metrics_v2 = (
    calculate_prompt_metrics(predictions_v2)
)

summary_comparison = pd.DataFrame(
    {
        "V1": summary_v1,
        "V2": summary_v2,
    }
)

target_comparison = pd.DataFrame(
    {
        "V1_positive_precision":
            metrics_v1["positive_precision"],
        "V2_positive_precision":
            metrics_v2["positive_precision"],
        "V1_positive_coverage":
            metrics_v1["positive_gold_coverage"],
        "V2_positive_coverage":
            metrics_v2["positive_gold_coverage"],
        "V1_negative_precision":
            metrics_v1["negative_precision"],
        "V2_negative_precision":
            metrics_v2["negative_precision"],
        "V1_negative_coverage":
            metrics_v1["negative_gold_coverage"],
        "V2_negative_coverage":
            metrics_v2["negative_gold_coverage"],
        "V1_neutral_auc":
            metrics_v1["neutral_unknown_auc"],
        "V2_neutral_auc":
            metrics_v2["neutral_unknown_auc"],
    }
)

status_comparison = (
    predictions_v1[
        [
            "study_uid",
            "target",
            "status",
        ]
    ]
    .rename(
        columns={
            "status": "V1_status",
        }
    )
    .merge(
        predictions_v2[
            [
                "study_uid",
                "target",
                "status",
            ]
        ].rename(
            columns={
                "status": "V2_status",
            }
        ),
        on=[
            "study_uid",
            "target",
        ],
        validate="one_to_one",
    )
)

transition_table = pd.crosstab(
    status_comparison["V1_status"],
    status_comparison["V2_status"],
)

print("Prompt summary comparison:")
display(
    summary_comparison.style.format("{:.4f}")
)

print("\nPer-target comparison:")
display(
    target_comparison.style.format("{:.3f}")
)

print("\nStatus transitions from V1 to V2:")
display(transition_table)

In [ ]:
import hashlib

FROZEN_PROMPT_NAME = "conservative_v2"
FROZEN_PROMPT_SHA256 = hashlib.sha256(
    system_prompt_v2.encode("utf-8")
).hexdigest()

frozen_extractor_config = {
    "prompt_name": FROZEN_PROMPT_NAME,
    "prompt_sha256": FROZEN_PROMPT_SHA256,
    "model_id": EXPECTED_MODEL_ID,
    "model_revision": EXPECTED_REVISION,
    "split_seed": SPLIT_SEED,
    "status_values": sorted(STATUS_VALUES),
    "severity_values": sorted(SEVERITY_VALUES),
    "positive_supervision": ["present"],
    "negative_supervision": ["absent"],
    "masked_supervision": [
        "uncertain",
        "not_mentioned",
    ],
    "negated_field_overrides_status": False,
}

frozen_config_path = Path(
    "/kaggle/working/frozen_report_extractor_config.json"
)

frozen_config_path.write_text(
    json.dumps(
        frozen_extractor_config,
        indent=2,
    ),
    encoding="utf-8",
)

evaluation_row_indices = gold_indices[
    best_evaluation_positions
]

evaluation_output_path = Path(
    "/kaggle/working/evaluation_report_extractions.jsonl"
)

EVALUATION_BATCH_SIZE = 2
EVALUATION_MAX_NEW_TOKENS = 700

evaluation_records = []
evaluation_start = time.perf_counter()

for batch_start in range(
    0,
    len(evaluation_row_indices),
    EVALUATION_BATCH_SIZE,
):
    batch_indices = evaluation_row_indices[
        batch_start:
        batch_start + EVALUATION_BATCH_SIZE
    ]

    conversations = [
        [
            {
                "role": "system",
                "content": system_prompt_v2,
            },
            {
                "role": "user",
                "content": (
                    "Extract the twelve target states "
                    "from this report:\n\n"
                    + reports.loc[row_index]
                ),
            },
        ]
        for row_index in batch_indices
    ]

    batch_inputs = tokenizer.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        padding=True,
        return_dict=True,
        return_tensors="pt",
    ).to(report_model.device)

    input_width = batch_inputs["input_ids"].shape[1]

    batch_start_time = time.perf_counter()

    with torch.inference_mode():
        batch_generated = report_model.generate(
            **batch_inputs,
            max_new_tokens=EVALUATION_MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            use_cache=True,
        )

    torch.cuda.synchronize()

    batch_seconds = (
        time.perf_counter()
        - batch_start_time
    )

    for local_index, row_index in enumerate(
        batch_indices
    ):
        response_text = tokenizer.decode(
            batch_generated[
                local_index,
                input_width:,
            ],
            skip_special_tokens=True,
        ).strip()

        record = {
            "study_uid": str(
                train.loc[
                    row_index,
                    "StudyInstanceUID",
                ]
            ),
            "script": str(
                dominant_script.loc[
                    row_index
                ]
            ),
            "predictions": None,
            "corrections": [],
            "error": None,
        }

        try:
            normalized, corrections = (
                parse_report_response(
                    response_text
                )
            )

            record["predictions"] = normalized
            record["corrections"] = corrections

        except Exception as error:
            record["error"] = (
                f"{type(error).__name__}: {error}"
            )

            debug_path = (
                debug_directory
                / (
                    "evaluation_"
                    f"{batch_start + local_index:03d}.txt"
                )
            )

            debug_path.write_text(
                response_text,
                encoding="utf-8",
            )

        evaluation_records.append(record)

    with evaluation_output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for record in evaluation_records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    completed = min(
        batch_start + len(batch_indices),
        len(evaluation_row_indices),
    )

    print(
        f"Processed {completed:02d}/"
        f"{len(evaluation_row_indices):02d} "
        f"in {batch_seconds:.1f} seconds"
    )

    del batch_inputs
    del batch_generated
    torch.cuda.empty_cache()

evaluation_seconds = (
    time.perf_counter()
    - evaluation_start
)

successful_evaluation_records = [
    record
    for record in evaluation_records
    if record["error"] is None
]

failed_evaluation_records = [
    record
    for record in evaluation_records
    if record["error"] is not None
]

evaluation_corrections = sum(
    len(record["corrections"])
    for record in successful_evaluation_records
)

print()
print("Frozen prompt:", FROZEN_PROMPT_NAME)
print("Prompt SHA256:", FROZEN_PROMPT_SHA256)
print(
    "Evaluation time (minutes):",
    round(evaluation_seconds / 60, 2),
)
print(
    "Successful reports:",
    len(successful_evaluation_records),
)
print(
    "Failed reports:",
    len(failed_evaluation_records),
)
print(
    "Semantic corrections:",
    evaluation_corrections,
)
print("Config:", frozen_config_path)
print("Checkpoint:", evaluation_output_path)

In [ ]:
from collections import Counter

evaluation_error_summary = Counter(
    record["error"]
    for record in failed_evaluation_records
)

evaluation_failure_rows = []

for position, record in enumerate(
    evaluation_records
):
    if record["error"] is None:
        continue

    debug_path = (
        debug_directory
        / f"evaluation_{position:03d}.txt"
    )

    evaluation_failure_rows.append(
        {
            "position": position,
            "script": record["script"],
            "response_characters": (
                debug_path.stat().st_size
                if debug_path.exists()
                else None
            ),
            "error": record["error"],
        }
    )

print("Evaluation error summary:")

for error, count in evaluation_error_summary.items():
    print(f"- {count} × {error}")

print("\nFailed response diagnostics:")
display(pd.DataFrame(evaluation_failure_rows))

In [ ]:
SEVERITY_ALIASES.update(
    {
        "low_grade": "mild",
        "intermediate_grade": "moderate",
        "high_grade": "severe",
        "grade_1": "mild",
        "grade_i": "mild",
        "grade_2": "moderate",
        "grade_ii": "moderate",
        "grade_3": "severe",
        "grade_iii": "severe",
    }
)

recovered_evaluation_count = 0

for position, record in enumerate(
    evaluation_records
):
    if record["error"] is None:
        continue

    debug_path = (
        debug_directory
        / f"evaluation_{position:03d}.txt"
    )

    response_text = debug_path.read_text(
        encoding="utf-8"
    )

    normalized, corrections = (
        parse_report_response(response_text)
    )

    record["predictions"] = normalized
    record["corrections"] = corrections
    record["error"] = None

    recovered_evaluation_count += 1

successful_evaluation_records = [
    record
    for record in evaluation_records
    if record["error"] is None
]

failed_evaluation_records = [
    record
    for record in evaluation_records
    if record["error"] is not None
]

with evaluation_output_path.open(
    "w",
    encoding="utf-8",
) as file:
    for record in evaluation_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

frozen_extractor_config[
    "severity_aliases"
] = SEVERITY_ALIASES

frozen_config_path.write_text(
    json.dumps(
        frozen_extractor_config,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Recovered evaluation reports:",
    recovered_evaluation_count,
)
print(
    "Successful evaluation reports:",
    len(successful_evaluation_records),
)
print(
    "Failed evaluation reports:",
    len(failed_evaluation_records),
)
print(
    "Total semantic corrections:",
    sum(
        len(record["corrections"])
        for record in successful_evaluation_records
    ),
)

In [ ]:
evaluation_predictions = (
    records_to_prediction_frame(
        successful_evaluation_records
    )
)

evaluation_summary, evaluation_target_metrics = (
    calculate_prompt_metrics(
        evaluation_predictions
    )
)

development_evaluation_summary = pd.DataFrame(
    {
        "development": summary_v2,
        "evaluation": evaluation_summary,
    }
)

evaluation_script_metrics = (
    evaluation_predictions
    .groupby("script")
    .apply(
        lambda frame: pd.Series(
            {
                "decisions": len(frame),
                "resolved": int(
                    frame["is_resolved"].sum()
                ),
                "resolved_coverage": float(
                    frame["is_resolved"].mean()
                ),
                "resolved_accuracy": float(
                    (
                        frame.loc[
                            frame["is_resolved"],
                            "gold_label",
                        ]
                        == (
                            frame.loc[
                                frame["is_resolved"],
                                "status",
                            ]
                            == "present"
                        ).astype(int)
                    ).mean()
                ),
            }
        ),
        include_groups=False,
    )
)

print("Frozen extractor: development versus evaluation")
display(
    development_evaluation_summary.style.format(
        "{:.4f}"
    )
)

print("\nLocked evaluation metrics by target:")
display(
    evaluation_target_metrics.style.format(
        {
            "positive_precision": "{:.3f}",
            "negative_precision": "{:.3f}",
            "positive_gold_coverage": "{:.3f}",
            "negative_gold_coverage": "{:.3f}",
            "resolved_coverage": "{:.3f}",
            "unknown_rate": "{:.3f}",
            "neutral_unknown_auc": "{:.3f}",
        }
    )
)

print("\nLocked evaluation metrics by writing system:")
display(
    evaluation_script_metrics.style.format(
        {
            "resolved_coverage": "{:.3f}",
            "resolved_accuracy": "{:.3f}",
        }
    )
)

print(
    "\nSemantic correction rate:",
    round(
        sum(
            len(record["corrections"])
            for record in successful_evaluation_records
        )
        / (
            len(successful_evaluation_records)
            * len(TARGETS)
        ),
        4,
    ),
)

In [ ]:
BENCHMARK_BATCH_SIZES = [
    2,
    4,
    8,
]

BENCHMARK_REPORT_COUNT = 8
BENCHMARK_MAX_NEW_TOKENS = 700

development_lengths = (
    reports.loc[
        development_row_indices
    ]
    .str.len()
    .sort_values()
)

length_positions = np.linspace(
    0,
    len(development_lengths) - 1,
    BENCHMARK_REPORT_COUNT,
    dtype=int,
)

benchmark_row_indices = (
    development_lengths
    .iloc[length_positions]
    .index
    .to_numpy()
)

v2_reference_by_uid = {
    record["study_uid"]: record["predictions"]
    for record in successful_records_v2
}

benchmark_rows = []

for batch_size in BENCHMARK_BATCH_SIZES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    predicted_by_uid = {}
    parse_errors = 0
    out_of_memory = False

    benchmark_start = time.perf_counter()

    try:
        for batch_start in range(
            0,
            len(benchmark_row_indices),
            batch_size,
        ):
            batch_indices = benchmark_row_indices[
                batch_start:
                batch_start + batch_size
            ]

            conversations = [
                [
                    {
                        "role": "system",
                        "content": system_prompt_v2,
                    },
                    {
                        "role": "user",
                        "content": (
                            "Extract the twelve target states "
                            "from this report:\n\n"
                            + reports.loc[row_index]
                        ),
                    },
                ]
                for row_index in batch_indices
            ]

            batch_inputs = tokenizer.apply_chat_template(
                conversations,
                add_generation_prompt=True,
                tokenize=True,
                padding=True,
                return_dict=True,
                return_tensors="pt",
            ).to(report_model.device)

            input_width = (
                batch_inputs["input_ids"].shape[1]
            )

            with torch.inference_mode():
                batch_generated = report_model.generate(
                    **batch_inputs,
                    max_new_tokens=(
                        BENCHMARK_MAX_NEW_TOKENS
                    ),
                    do_sample=False,
                    temperature=None,
                    top_p=None,
                    top_k=None,
                    use_cache=True,
                )

            for local_index, row_index in enumerate(
                batch_indices
            ):
                response_text = tokenizer.decode(
                    batch_generated[
                        local_index,
                        input_width:,
                    ],
                    skip_special_tokens=True,
                ).strip()

                study_uid = str(
                    train.loc[
                        row_index,
                        "StudyInstanceUID",
                    ]
                )

                try:
                    normalized, _ = (
                        parse_report_response(
                            response_text
                        )
                    )

                    predicted_by_uid[
                        study_uid
                    ] = normalized

                except Exception:
                    parse_errors += 1

            del batch_inputs
            del batch_generated

        torch.cuda.synchronize()

    except torch.OutOfMemoryError:
        out_of_memory = True

    elapsed = (
        time.perf_counter()
        - benchmark_start
    )

    mismatched_reports = 0

    if not out_of_memory:
        for row_index in benchmark_row_indices:
            study_uid = str(
                train.loc[
                    row_index,
                    "StudyInstanceUID",
                ]
            )

            if (
                predicted_by_uid.get(study_uid)
                != v2_reference_by_uid.get(study_uid)
            ):
                mismatched_reports += 1

    completed_reports = len(predicted_by_uid)

    seconds_per_report = (
        elapsed / completed_reports
        if completed_reports
        else float("nan")
    )

    benchmark_rows.append(
        {
            "batch_size": batch_size,
            "completed_reports": completed_reports,
            "out_of_memory": out_of_memory,
            "parse_errors": parse_errors,
            "mismatched_reports": mismatched_reports,
            "elapsed_seconds": elapsed,
            "seconds_per_report": seconds_per_report,
            "projected_4407_hours": (
                seconds_per_report
                * 4407
                / 3600
            ),
            "peak_gpu_memory_gb": (
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
        }
    )

    torch.cuda.empty_cache()

benchmark_results = pd.DataFrame(
    benchmark_rows
)

display(
    benchmark_results.style.format(
        {
            "elapsed_seconds": "{:.2f}",
            "seconds_per_report": "{:.2f}",
            "projected_4407_hours": "{:.2f}",
            "peak_gpu_memory_gb": "{:.2f}",
        }
    )
)

In [ ]:
print("CUDA device count:", torch.cuda.device_count())

for device_index in range(
    torch.cuda.device_count()
):
    properties = torch.cuda.get_device_properties(
        device_index
    )

    print(
        f"GPU {device_index}:",
        properties.name,
        "-",
        round(
            properties.total_memory / 1024**3,
            2,
        ),
        "GB",
    )

print(
    "Allocated on GPU 0:",
    round(
        torch.cuda.memory_allocated(0) / 1024**3,
        2,
    ),
    "GB",
)

if torch.cuda.device_count() > 1:
    print(
        "Allocated on GPU 1:",
        round(
            torch.cuda.memory_allocated(1) / 1024**3,
            2,
        ),
        "GB",
    )

In [ ]:
from concurrent.futures import ThreadPoolExecutor

torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)

gpu1_load_start = time.perf_counter()

report_model_gpu1 = (
    AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_DIR,
        local_files_only=True,
        dtype=torch.float16,
        device_map={"": 1},
        low_cpu_mem_usage=True,
    )
)

report_model_gpu1.eval()
report_model_gpu1.requires_grad_(False)

torch.cuda.synchronize(1)

print(
    "GPU 1 model load time (seconds):",
    round(
        time.perf_counter() - gpu1_load_start,
        2,
    ),
)
print(
    "GPU 1 allocated memory (GB):",
    round(
        torch.cuda.memory_allocated(1) / 1024**3,
        2,
    ),
)

DUAL_GPU_REPORT_COUNT = 16
PER_GPU_BATCH_SIZE = 8

development_lengths = (
    reports.loc[
        development_row_indices
    ]
    .str.len()
    .sort_values()
)

selected_positions = np.linspace(
    0,
    len(development_lengths) - 1,
    DUAL_GPU_REPORT_COUNT,
    dtype=int,
)

dual_gpu_row_indices = (
    development_lengths
    .iloc[selected_positions]
    .index
    .to_numpy()
)

# Alternate sorted report lengths between GPUs to balance work.
gpu0_indices = dual_gpu_row_indices[0::2]
gpu1_indices = dual_gpu_row_indices[1::2]

def prepare_batch(row_indices, device_index):
    conversations = [
        [
            {
                "role": "system",
                "content": system_prompt_v2,
            },
            {
                "role": "user",
                "content": (
                    "Extract the twelve target states "
                    "from this report:\n\n"
                    + reports.loc[row_index]
                ),
            },
        ]
        for row_index in row_indices
    ]

    inputs = tokenizer.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        padding=True,
        return_dict=True,
        return_tensors="pt",
    )

    return inputs.to(
        torch.device(
            f"cuda:{device_index}"
        )
    )

gpu0_inputs = prepare_batch(
    gpu0_indices,
    0,
)

gpu1_inputs = prepare_batch(
    gpu1_indices,
    1,
)

def generate_on_gpu(
    model,
    model_inputs,
    device_index,
):
    input_width = model_inputs[
        "input_ids"
    ].shape[1]

    with torch.cuda.device(device_index):
        with torch.inference_mode():
            generated = model.generate(
                **model_inputs,
                max_new_tokens=700,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                use_cache=True,
            )

        torch.cuda.synchronize(
            device_index
        )

    return (
        generated.cpu(),
        input_width,
    )

dual_gpu_start = time.perf_counter()

with ThreadPoolExecutor(
    max_workers=2
) as executor:
    gpu0_future = executor.submit(
        generate_on_gpu,
        report_model,
        gpu0_inputs,
        0,
    )

    gpu1_future = executor.submit(
        generate_on_gpu,
        report_model_gpu1,
        gpu1_inputs,
        1,
    )

    gpu0_generated, gpu0_width = (
        gpu0_future.result()
    )

    gpu1_generated, gpu1_width = (
        gpu1_future.result()
    )

dual_gpu_seconds = (
    time.perf_counter()
    - dual_gpu_start
)

dual_gpu_predictions = {}
dual_gpu_parse_errors = 0

for (
    row_indices,
    generated,
    input_width,
) in [
    (
        gpu0_indices,
        gpu0_generated,
        gpu0_width,
    ),
    (
        gpu1_indices,
        gpu1_generated,
        gpu1_width,
    ),
]:
    for local_index, row_index in enumerate(
        row_indices
    ):
        response_text = tokenizer.decode(
            generated[
                local_index,
                input_width:,
            ],
            skip_special_tokens=True,
        ).strip()

        study_uid = str(
            train.loc[
                row_index,
                "StudyInstanceUID",
            ]
        )

        try:
            normalized, _ = (
                parse_report_response(
                    response_text
                )
            )

            dual_gpu_predictions[
                study_uid
            ] = normalized

        except Exception:
            dual_gpu_parse_errors += 1

dual_gpu_mismatches = 0

for row_index in dual_gpu_row_indices:
    study_uid = str(
        train.loc[
            row_index,
            "StudyInstanceUID",
        ]
    )

    if (
        dual_gpu_predictions.get(study_uid)
        != v2_reference_by_uid.get(study_uid)
    ):
        dual_gpu_mismatches += 1

seconds_per_report = (
    dual_gpu_seconds
    / len(dual_gpu_row_indices)
)

projected_hours = (
    seconds_per_report
    * len(train)
    / 3600
)

print()
print(
    "Reports processed:",
    len(dual_gpu_row_indices),
)
print(
    "Total concurrent time (seconds):",
    round(dual_gpu_seconds, 2),
)
print(
    "Effective seconds per report:",
    round(seconds_per_report, 2),
)
print(
    "Projected 4,407-report runtime (hours):",
    round(projected_hours, 2),
)
print(
    "Parse errors:",
    dual_gpu_parse_errors,
)
print(
    "Prediction mismatches:",
    dual_gpu_mismatches,
)
print(
    "Peak GPU 0 memory (GB):",
    round(
        torch.cuda.max_memory_allocated(0)
        / 1024**3,
        2,
    ),
)
print(
    "Peak GPU 1 memory (GB):",
    round(
        torch.cuda.max_memory_allocated(1)
        / 1024**3,
        2,
    ),
)

del gpu0_inputs
del gpu1_inputs
del gpu0_generated
del gpu1_generated

torch.cuda.empty_cache()

In [ ]:
difference_rows = []

for row_index in dual_gpu_row_indices:
    study_uid = str(
        train.loc[
            row_index,
            "StudyInstanceUID",
        ]
    )

    reference = v2_reference_by_uid[
        study_uid
    ]

    candidate = dual_gpu_predictions.get(
        study_uid
    )

    if candidate is None:
        continue

    for target in TARGETS:
        for field in [
            "status",
            "severity",
            "negated",
        ]:
            if (
                reference[target][field]
                != candidate[target][field]
            ):
                difference_rows.append(
                    {
                        "target": target,
                        "field": field,
                        "reference": (
                            reference[target][field]
                        ),
                        "dual_gpu": (
                            candidate[target][field]
                        ),
                        "gold_label": int(
                            train.loc[
                                row_index,
                                target,
                            ]
                        ),
                    }
                )

difference_frame = pd.DataFrame(
    difference_rows
)

print(
    "Reports with any difference:",
    dual_gpu_mismatches,
)
print(
    "Total field differences:",
    len(difference_frame),
)

if len(difference_frame):
    print("\nDifferences by field:")
    display(
        difference_frame[
            "field"
        ]
        .value_counts()
        .rename("count")
        .to_frame()
    )

    print("\nDifferences by target and field:")
    display(
        difference_frame
        .groupby(
            [
                "target",
                "field",
            ]
        )
        .size()
        .rename("count")
        .to_frame()
    )

    status_differences = difference_frame[
        difference_frame["field"] == "status"
    ]

    if len(status_differences):
        print("\nStatus transitions:")
        display(
            pd.crosstab(
                status_differences["reference"],
                status_differences["dual_gpu"],
            )
        )

        print("\nStatus differences with gold labels:")
        display(
            status_differences[
                [
                    "target",
                    "reference",
                    "dual_gpu",
                    "gold_label",
                ]
            ]
        )